In [1]:
# Cell 1 — Imports  (pyarrow MUST be first: Windows DLL fix)
import pyarrow
import transformers
import torch
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm

print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

torch 2.5.1+cu121 | CUDA: True


In [2]:
# Cell 2 — Config
from dotenv import load_dotenv
load_dotenv('../.env')

MODEL_NAME     = 'meta-llama/Llama-3.1-8B-Instruct'
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
N_SAMPLES      = 200
NLDD_THRESHOLD = 0.1
PROBE_LAYERS   = [8, 12, 16, 20, 24, 28, 31]
PCA_COMPONENTS = 50
OUT_DIR        = '../outputs'
PROBE_PATH     = os.path.join(OUT_DIR, 'probe_llama.pkl')

os.makedirs(OUT_DIR, exist_ok=True)
print(f'Model: {MODEL_NAME}  |  Device: {DEVICE}')
print(f'Probe layers: {PROBE_LAYERS}')
print(f'PCA components: {PCA_COMPONENTS}  (d_model=4096 → {PCA_COMPONENTS} dims)')
print(f'Probe will be saved to: {PROBE_PATH}')

Model: meta-llama/Llama-3.1-8B-Instruct  |  Device: cuda
Probe layers: [8, 12, 16, 20, 24, 28, 31]
PCA components: 50  (d_model=4096 → 50 dims)
Probe will be saved to: ../outputs\probe_llama.pkl


In [3]:
# Cell 3 — Load LLaMA-3.1-8B and GSM8K samples
from src.utils.model_loader import get_model
from data.dataset_loader import load_gsm8k

print(f'Loading {MODEL_NAME} ...')
model = get_model(MODEL_NAME, device=DEVICE)
print(f'Loaded  layers={model.cfg.n_layers}  d_model={model.cfg.d_model}')

samples = load_gsm8k(split='test', max_samples=N_SAMPLES)
print(f'Loaded {len(samples)} GSM8K samples')

Loading meta-llama/Llama-3.1-8B-Instruct ...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded  layers=32  d_model=4096


Loaded 200 GSM8K samples


In [4]:
# Cell 4 — Generate faithfulness labels and activations at PROBE_LAYERS
from src.probing.faithfulness_detector import generate_faithfulness_labels

def extract_at_layers(model, prompt, layers):
    """Extract hook_resid_post only at specified layers (efficient for large models)."""
    tokens = model.to_tokens(prompt)
    activations = {}
    hooks = [
        (
            f'blocks.{l}.hook_resid_post',
            lambda act, hook, ll=l: activations.__setitem__(ll, act[0].detach().cpu()),
        )
        for l in layers
    ]
    with torch.no_grad():
        model.run_with_hooks(tokens, fwd_hooks=hooks)
    return activations

per_step_acts   = {l: [] for l in PROBE_LAYERS}
per_step_labels = []

for sample in tqdm(samples, desc='Extracting activations'):
    question, steps = sample['question'], sample['steps']
    if len(steps) < 2:
        continue
    prompt = 'Q: ' + question + ' A: ' + ' '.join(steps)
    try:
        label_tuples = generate_faithfulness_labels(
            model, prompt, steps,
            strategy='number_perturbation', nldd_threshold=NLDD_THRESHOLD)
    except Exception:
        continue
    acts = extract_at_layers(model, prompt, PROBE_LAYERS)
    for _, is_faithful, _ in label_tuples:
        for l in PROBE_LAYERS:
            per_step_acts[l].append(acts[l].float().mean(dim=0).numpy())
        per_step_labels.append(is_faithful)

print(f'Per-step samples: {len(per_step_labels)}')
print(f'Faithful fraction: {sum(per_step_labels)/max(1,len(per_step_labels)):.1%}')

Extracting activations:   0%|          | 0/200 [00:00<?, ?it/s]

Extracting activations:   0%|          | 1/200 [00:02<07:42,  2.33s/it]

Extracting activations:   1%|          | 2/200 [00:03<06:00,  1.82s/it]

Extracting activations:   2%|▏         | 3/200 [00:06<06:38,  2.02s/it]

Extracting activations:   2%|▏         | 4/200 [00:07<05:53,  1.80s/it]

...

Extracting activations:  99%|█████████▉| 198/200 [07:09<00:03,  1.66s/it]

Extracting activations: 100%|█████████▉| 199/200 [07:12<00:02,  2.02s/it]

Extracting activations: 100%|██████████| 200/200 [07:15<00:00,  2.28s/it]

Extracting activations: 100%|██████████| 200/200 [07:15<00:00,  2.18s/it]

Per-step samples: 897
Faithful fraction: 41.4%


In [5]:
# Cell 5 — Train probe at each PROBE_LAYER (raw + PCA-50), Figure 5a
from src.probing.faithfulness_detector import FaithfulnessProbe
from src.utils.visualizer import plot_probe_accuracy_by_layer

layer_accuracies_raw = {}
layer_accuracies_pca = {}
probes = {}

if len(per_step_labels) >= 4 and len(set(per_step_labels)) >= 2:
    print(f'Training probes at layers {PROBE_LAYERS} — raw (4096-dim) and PCA-{PCA_COMPONENTS}...')
    print(f'\n{"Layer":>6}  {"Raw CV":>8}  {"PCA-50 CV":>10}')
    print('  ' + '-' * 28)
    for l in tqdm(PROBE_LAYERS, desc='Layer probes', leave=False):
        acts_arr = np.array(per_step_acts[l])

        # Raw probe (no dimensionality reduction)
        probe_raw = FaithfulnessProbe(layer=l)
        cv_raw = probe_raw.train(acts_arr, per_step_labels)
        layer_accuracies_raw[l] = cv_raw

        # PCA-reduced probe (4096 → PCA_COMPONENTS)
        probe_pca = FaithfulnessProbe(layer=l)
        cv_pca = probe_pca.train(acts_arr, per_step_labels, pca_components=PCA_COMPONENTS)
        layer_accuracies_pca[l] = cv_pca
        probes[l] = probe_pca  # PCA probe used for all downstream steps

        print(f'  Layer {l:2d}:  {cv_raw:.3f}     {cv_pca:.3f}')

    best_layer = max(layer_accuracies_pca, key=layer_accuracies_pca.get)
    print(f'\nBest layer (PCA-{PCA_COMPONENTS}): {best_layer}  '
          f'CV accuracy: {layer_accuracies_pca[best_layer]:.3f}')
    print(f'Best layer (raw):       {max(layer_accuracies_raw, key=layer_accuracies_raw.get)}  '
          f'CV accuracy: {max(layer_accuracies_raw.values()):.3f}')
else:
    best_layer = PROBE_LAYERS[len(PROBE_LAYERS) // 2]
    print(f'Insufficient data; using middle probe layer {best_layer}.')

if layer_accuracies_pca:
    fig = plot_probe_accuracy_by_layer(layer_accuracies_pca)
    axes = fig.get_axes()[0]
    axes.set_title(f'Figure 5a: Probe CV Accuracy by Layer (PCA-{PCA_COMPONENTS})')
    fig.savefig(os.path.join(OUT_DIR, 'figure5a_llama_probe_accuracy.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved -> outputs/figure5a_llama_probe_accuracy.png')

Training probes at layers [8, 12, 16, 20, 24, 28, 31] — raw (4096-dim) and PCA-50...

 Layer    Raw CV   PCA-50 CV
  ----------------------------


Layer probes:   0%|          | 0/7 [00:00<?, ?it/s]

Layer probes:  14%|█▍        | 1/7 [00:00<00:04,  1.33it/s]

  Layer  8:  0.602     0.610


Layer probes:  29%|██▊       | 2/7 [00:01<00:03,  1.32it/s]

...

Layer probes:  86%|████████▌ | 6/7 [00:05<00:00,  1.06it/s]

  Layer 28:  0.520     0.550


Layer probes: 100%|██████████| 7/7 [00:06<00:00,  1.16s/it]

  Layer 31:  0.516     0.556

Best layer (PCA-50): 8  CV accuracy: 0.610
Best layer (raw):       8  CV accuracy: 0.602
Saved -> outputs/figure5a_llama_probe_accuracy.png


C:\Users\gsr20\AppData\Local\Temp\ipykernel_8372\1487541450.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Cell 6 — Confusion matrix + confidence distribution (Figure 5b)
best_probe = probes.get(best_layer)
if best_probe is None:
    print('No probe trained; skipping Figure 5b.')
else:
    acts_arr = np.array(per_step_acts[best_layer])
    preds    = best_probe.predict(acts_arr)
    pred_labels = [lbl for lbl, _ in preds]
    confidences = [conf for _, conf in preds]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    cm = confusion_matrix(per_step_labels, pred_labels)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Unfaithful', 'Faithful'])
    disp.plot(ax=axes[0], colorbar=False)
    axes[0].set_title(f'Confusion Matrix \u2014 Layer {best_layer}')

    faith_conf = [c for c, l in zip(confidences, per_step_labels) if l]
    unfai_conf = [c for c, l in zip(confidences, per_step_labels) if not l]
    axes[1].hist(faith_conf, bins=10, alpha=0.65, color='steelblue', label='Faithful')
    axes[1].hist(unfai_conf, bins=10, alpha=0.65, color='salmon',    label='Unfaithful')
    axes[1].axvline(0.5, color='black', ls='--', lw=1, label='threshold=0.5')
    axes[1].set(xlabel='Probe confidence', ylabel='Count',
                title=f'Confidence Distribution \u2014 Layer {best_layer}')
    axes[1].legend(fontsize=8)

    fig.suptitle('Figure 5b: LLaMA Probe Evaluation', fontsize=12)
    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, 'figure5b_llama_probe_eval.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved -> outputs/figure5b_llama_probe_eval.png')

Saved -> outputs/figure5b_llama_probe_eval.png


C:\Users\gsr20\AppData\Local\Temp\ipykernel_8372\3727395627.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Cell 7 — PCA scatter + probe direction (Figure 5c)
from src.utils.visualizer import plot_pca_with_direction

if best_probe is not None:
    acts_arr  = np.array(per_step_acts[best_layer])
    direction = best_probe.direction
    fig = plot_pca_with_direction(acts_arr, direction)
    fig.set_size_inches(6, 6)
    fig.get_axes()[0].set_title(f'Figure 5c: PCA + Probe Direction (Layer {best_layer})')
    fig.savefig(os.path.join(OUT_DIR, 'figure5c_llama_pca.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved -> outputs/figure5c_llama_pca.png')
    print(f'Probe direction norm: {direction.norm().item():.4f}  (should be 1.0)')
else:
    print('No probe available; skipping Figure 5c.')

Saved -> outputs/figure5c_llama_pca.png
Probe direction norm: 1.0000  (should be 1.0)


C:\Users\gsr20\AppData\Local\Temp\ipykernel_8372\486994055.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Cell 8 — Save probe_llama.pkl
import json

if best_probe is not None:
    best_probe.save(PROBE_PATH)
    print(f'Probe saved: {PROBE_PATH}  (layer={best_probe.layer})')
    meta = {
        'best_layer': int(best_layer),
        'cv_accuracy_pca': float(layer_accuracies_pca.get(best_layer, 0)),
        'cv_accuracy_raw': float(layer_accuracies_raw.get(best_layer, 0)),
        'pca_components': PCA_COMPONENTS,
        'n_train': len(per_step_labels),
        'faithful_fraction': sum(per_step_labels) / max(1, len(per_step_labels)),
        'probe_layers': PROBE_LAYERS,
        'model': MODEL_NAME,
    }
    with open(os.path.join(OUT_DIR, 'probe_llama_meta.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    print('Probe metadata:', meta)
else:
    print('Warning: no probe trained; probe_llama.pkl not written.')

print('Notebook 05 complete')

Probe saved: ../outputs\probe_llama.pkl  (layer=8)
Probe metadata: {'best_layer': 8, 'cv_accuracy_pca': 0.6098013656114214, 'cv_accuracy_raw': 0.6019987585350715, 'pca_components': 50, 'n_train': 897, 'faithful_fraction': 0.4136008918617614, 'probe_layers': [8, 12, 16, 20, 24, 28, 31], 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
Notebook 05 complete
